# Bihar Scheduled Energy Flow Analysis

This notebook builds a Bihar-first exploratory workflow for daily scheduled energy and scheduled flow analysis from the uploaded scheduling and REA files.

Key guardrails used throughout:

- treat the outputs as scheduled energy / scheduled flows unless the source clearly proves otherwise
- keep Bihar whole as the default reporting level
- do not fabricate plant or entity mappings
- do not force a North Bihar geographic result where the data does not support it
- show routing diagnostics and ambiguity flags before relying on any sign convention
- keep mapping tables editable and easy to refine

Expected primary sources:

- `raw_data/Supporting_files_at_cr0226.xlsx` as the main daily schedule source
- `raw_data/REA-January-2026.xlsx` as the monthly reference / reconciliation source
- `raw_data/REA-January-2026.pdf` as terminology context only
- `raw_data/TP-NBPDCL_FY_2025-26.pdf` as optional context only

The tariff petition PDF is checked defensively because it may be absent from the local `raw_data/` folder.

In [1]:
from __future__ import annotations

from pathlib import Path
import importlib
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until the repository root containing src/ and config/ is found."""

    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/ and config/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.path_utils import ensure_project_directories, load_app_config, resolve_project_paths


REQUIRED_PACKAGES = {
    "openpyxl": "Excel workbook loading via pandas.read_excel",
    "matplotlib": "chart generation",
}
OPTIONAL_PACKAGES = {
    "PyPDF2": "optional PDF text inspection for reference documents",
}


def check_notebook_packages() -> None:
    """Fail early for required packages and warn for optional ones."""

    missing_required: list[str] = []
    for package_name, purpose in REQUIRED_PACKAGES.items():
        try:
            importlib.import_module(package_name)
        except ModuleNotFoundError:
            missing_required.append(f"{package_name} ({purpose})")

    if missing_required:
        raise ModuleNotFoundError(
            "Missing required packages for this notebook: " + ", ".join(missing_required)
        )

    for package_name, purpose in OPTIONAL_PACKAGES.items():
        try:
            importlib.import_module(package_name)
        except ModuleNotFoundError:
            warnings.warn(
                f"Optional package '{package_name}' is not installed. {purpose} will be skipped.",
                stacklevel=2,
            )


check_notebook_packages()

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"Project root: {PROJECT_ROOT}")

Project root: F:\Secure\CashFlowMgmt


In [2]:
APP_CONFIG = load_app_config(PROJECT_ROOT / "config" / "app_config.yaml")
PATHS = resolve_project_paths(cli_base_dir=PROJECT_ROOT, app_config=APP_CONFIG)
ensure_project_directories(PATHS)

OUTPUT_DIR = PATHS.processed_data_dir / "bihar_energy_flow"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "supporting": PATHS.raw_data_dir / "Supporting_files_at_cr0226.xlsx",
    "rea_xlsx": PATHS.raw_data_dir / "REA-January-2026.xlsx",
    "rea_pdf": PATHS.raw_data_dir / "REA-January-2026.pdf",
    "tp_nbpdcl_pdf": PATHS.raw_data_dir / "TP-NBPDCL_FY_2025-26.pdf",
}

print("Resolved paths:")
print(f"- raw_data_dir: {PATHS.raw_data_dir}")
print(f"- processed_data_dir: {PATHS.processed_data_dir}")
print(f"- output_dir: {OUTPUT_DIR}")
print()
print("File existence check:")
for label, path in RAW_FILES.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {label}: {status} -> {path}")

if not RAW_FILES["supporting"].exists() or not RAW_FILES["rea_xlsx"].exists():
    raise FileNotFoundError("The required Bihar Excel workbooks are missing from raw_data/.")

supporting_book = pd.ExcelFile(RAW_FILES["supporting"], engine="openpyxl")
rea_book = pd.ExcelFile(RAW_FILES["rea_xlsx"], engine="openpyxl")

print()
print("Supporting workbook sheets:")
print(supporting_book.sheet_names)
print()
print("REA workbook sheets:")
print(rea_book.sheet_names)

Resolved paths:
- raw_data_dir: F:\Secure\CashFlowMgmt\raw_data
- processed_data_dir: F:\Secure\CashFlowMgmt\data\processed
- output_dir: F:\Secure\CashFlowMgmt\data\processed\bihar_energy_flow

File existence check:
- supporting: FOUND -> F:\Secure\CashFlowMgmt\raw_data\Supporting_files_at_cr0226.xlsx
- rea_xlsx: FOUND -> F:\Secure\CashFlowMgmt\raw_data\REA-January-2026.xlsx
- rea_pdf: FOUND -> F:\Secure\CashFlowMgmt\raw_data\REA-January-2026.pdf
- tp_nbpdcl_pdf: MISSING -> F:\Secure\CashFlowMgmt\raw_data\TP-NBPDCL_FY_2025-26.pdf

Supporting workbook sheets:
['sch_at', 'sch_cr', 'd_urs_one_to_one']

REA workbook sheets:
['data (25)']


## 2. Data inspection

This section loads the Bihar schedule workbook, profiles each relevant sheet, identifies interval columns, prints routing-domain values, and samples the REA workbook structure for later reconciliation.

The goal here is visibility rather than premature cleaning. Any missing columns, odd headers, or ambiguous routing labels should stay visible.

In [3]:
ROUTING_COLUMNS = ["from_seb", "to_seb", "from_util", "to_util", "applicant", "link", "region"]
PRIMARY_SCHEDULE_SHEETS = ["sch_at", "sch_cr", "d_urs_one_to_one"]
ENTITY_PRIORITY_SOURCE = ["from_util", "from_seb", "applicant"]
ENTITY_PRIORITY_DESTINATION = ["to_util", "to_seb"]
PLANT_CANDIDATE_COLUMNS = ["from_util", "to_util", "applicant", "stn_code"]
REVIEW_FOCUS_NAMES = [
    "BARH-I",
    "BARH-II",
    "BARH",
    "NTPC (KBUNL-II)",
    "NABINAGAR POWER GENERATION COMPANY LIMITED",
    "NPGC",
    "BRBCL",
    "NTPC (BARAUNI-I)",
    "NTPC (BARAUNI-II)",
    "BSHPC",
    "BUXAR TPP UNIT #1",
    "BUXAR TPP UNIT #2",
    "EAST GANDAK",
    "TRIVENI",
    "SONE EASTERN",
    "SONE WESTERN",
]


def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    renamed = {col: str(col).strip().lower().replace(" ", "_") for col in frame.columns}
    return frame.rename(columns=renamed).copy()


INTERVAL_RE = re.compile(r"^energy\d{4}$", flags=re.IGNORECASE)


def identify_interval_columns(columns: list[str] | pd.Index) -> list[str]:
    interval_cols = [col for col in columns if INTERVAL_RE.match(str(col))]
    return sorted(interval_cols, key=lambda value: int(str(value)[6:]))


EXCEL_EPOCH = pd.Timestamp("1899-12-30")


def coerce_schedule_date(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    parsed_numeric = pd.to_datetime(numeric, unit="D", origin=EXCEL_EPOCH, errors="coerce")
    parsed_string = pd.to_datetime(series, errors="coerce", dayfirst=False)
    combined = parsed_numeric.where(parsed_numeric.notna(), parsed_string)
    return combined.dt.normalize()



def coalesce_first_non_blank(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    result = pd.Series(pd.NA, index=frame.index, dtype="object")
    for column in columns:
        if column not in frame.columns:
            continue
        candidate = frame[column].where(frame[column].notna(), pd.NA).astype("string").str.strip()
        candidate = candidate.where(candidate.ne(""), pd.NA)
        result = result.where(result.notna(), candidate)
    return result



def standardize_text(value: object) -> str | None:
    if value is None or pd.isna(value):
        return None
    cleaned = re.sub(r"\s+", " ", str(value).strip())
    return cleaned.upper() if cleaned else None



def load_excel_sheets(path: Path, *, header: int | None = 0, only_sheets: list[str] | None = None) -> dict[str, pd.DataFrame]:
    loaded = pd.read_excel(path, sheet_name=only_sheets or None, header=header, engine="openpyxl")
    if isinstance(loaded, pd.DataFrame):
        loaded = {str(only_sheets[0] if only_sheets else "sheet1"): loaded}
    return {sheet_name: normalize_columns(df) for sheet_name, df in loaded.items()}



def print_sheet_profile(sheet_name: str, frame: pd.DataFrame) -> None:
    interval_cols = identify_interval_columns(frame.columns)
    print(f"Sheet: {sheet_name}")
    print(f"- rows: {len(frame):,}")
    print(f"- columns: {len(frame.columns):,}")
    print(f"- interval columns detected: {len(interval_cols)}")
    print(f"- first columns: {frame.columns[:12].tolist()}")
    display(frame.head(3))
    print()



def print_unique_values(frame: pd.DataFrame, columns: list[str], max_values: int = 40) -> None:
    for column in columns:
        if column not in frame.columns:
            print(f"Column '{column}' not present in this sheet.")
            continue
        unique_values = (
            frame[column].dropna().astype("string").str.strip().replace("", pd.NA).dropna().drop_duplicates().sort_values().tolist()
        )
        print(f"Unique values for {column} ({len(unique_values)} shown up to {max_values}):")
        print(unique_values[:max_values])
        print()



def prepare_schedule_sheet(frame: pd.DataFrame, sheet_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    prepared = frame.copy()
    prepared = pd.concat(
        [
            prepared,
            pd.DataFrame(
                {
                    "source_sheet": sheet_name,
                    "source_row_number": np.arange(2, len(prepared) + 2),
                    "sch_date": coerce_schedule_date(prepared["sch_date"]) if "sch_date" in prepared.columns else pd.NaT,
                },
                index=prepared.index,
            ),
        ],
        axis=1,
    )
    prepared["row_uid"] = prepared["source_sheet"] + "::" + prepared["source_row_number"].astype(str)

    interval_cols = identify_interval_columns(prepared.columns)
    if not interval_cols:
        raise ValueError(f"No ENERGY#### columns were found in sheet '{sheet_name}'.")

    prepared[interval_cols] = prepared[interval_cols].apply(pd.to_numeric, errors="coerce")
    prepared = pd.concat(
        [
            prepared,
            pd.DataFrame(
                {
                    "daily_total_raw": prepared[interval_cols].sum(axis=1, min_count=1),
                    "daily_total_abs": prepared[interval_cols].sum(axis=1, min_count=1).abs(),
                    "source_entity_raw": coalesce_first_non_blank(prepared, ENTITY_PRIORITY_SOURCE),
                    "destination_entity_raw": coalesce_first_non_blank(prepared, ENTITY_PRIORITY_DESTINATION),
                },
                index=prepared.index,
            ),
        ],
        axis=1,
    ).copy()

    long_frame = prepared.melt(
        id_vars=[col for col in prepared.columns if col not in interval_cols],
        value_vars=interval_cols,
        var_name="interval_label",
        value_name="energy_mwh",
    )
    long_frame["energy_mwh"] = pd.to_numeric(long_frame["energy_mwh"], errors="coerce")
    long_frame["interval_label"] = long_frame["interval_label"].str.upper()
    interval_time = long_frame["interval_label"].str.replace("ENERGY", "", regex=False)
    long_frame["interval_start"] = pd.to_datetime(
        long_frame["sch_date"].dt.strftime("%Y-%m-%d") + " " + interval_time,
        format="%Y-%m-%d %H%M",
        errors="coerce",
    )
    return prepared, long_frame



def infer_entity_type(raw_entity: str) -> str:
    entity_upper = standardize_text(raw_entity) or ""
    if entity_upper in {"NBPDCL", "SBPDCL", "BSPHCL", "BSEB", "BIHAR"}:
        return "state_utility_or_discom"
    if entity_upper in {"ER", "WR", "NR", "SR", "NER"}:
        return "regional_code"
    if entity_upper in {"IEX", "PXI", "PXIL", "HPX"}:
        return "power_exchange"
    if "-" in entity_upper and len(entity_upper) <= 12:
        return "routing_link_code"
    if not entity_upper:
        return "missing"
    return "entity_unreviewed"



def seed_entity_mapping(schedule_rows: pd.DataFrame) -> pd.DataFrame:
    raw_entities: set[str] = set()
    for column in ROUTING_COLUMNS:
        if column not in schedule_rows.columns:
            continue
        values = schedule_rows[column].dropna().astype("string").str.strip().replace("", pd.NA).dropna().tolist()
        raw_entities.update(values)

    seeded_rows: list[dict[str, object]] = []
    for raw_entity in sorted(raw_entities, key=lambda value: standardize_text(value) or ""):
        entity_upper = standardize_text(raw_entity) or ""
        belongs_to_bihar = pd.NA
        belongs_to_nbpdcl = pd.NA
        belongs_to_sbpdcl = pd.NA
        belongs_to_interstate = pd.NA
        notes: list[str] = []

        if any(token in entity_upper for token in ["BIHAR", "BSPHCL", "BSEB"]):
            belongs_to_bihar = True
            notes.append("Auto-seeded from explicit Bihar/BSPHCL/BSEB text.")
        if "NBPDCL" in entity_upper:
            belongs_to_bihar = True
            belongs_to_nbpdcl = True
            belongs_to_sbpdcl = False
            notes.append("Auto-seeded from explicit NBPDCL text.")
        if "SBPDCL" in entity_upper:
            belongs_to_bihar = True
            belongs_to_sbpdcl = True
            belongs_to_nbpdcl = False
            notes.append("Auto-seeded from explicit SBPDCL text.")
        if entity_upper in {"IEX", "PXI", "PXIL", "HPX", "ER", "WR", "NR", "SR", "NER"}:
            belongs_to_interstate = True
            notes.append("Auto-seeded from explicit market or regional routing token.")
        if not notes:
            notes.append("Unreviewed candidate entity from schedule routing fields.")

        seeded_rows.append(
            {
                "raw_entity": raw_entity,
                "entity_type": infer_entity_type(raw_entity),
                "belongs_to_bihar": belongs_to_bihar,
                "belongs_to_nbpdcl": belongs_to_nbpdcl,
                "belongs_to_sbpdcl": belongs_to_sbpdcl,
                "belongs_to_interstate": belongs_to_interstate,
                "notes": " ".join(notes),
            }
        )

    return pd.DataFrame(seeded_rows)

In [4]:
supporting_sheets = load_excel_sheets(RAW_FILES["supporting"], only_sheets=PRIMARY_SCHEDULE_SHEETS)
rea_reference_raw = pd.read_excel(
    RAW_FILES["rea_xlsx"],
    sheet_name=rea_book.sheet_names[0],
    header=None,
    engine="openpyxl",
)

print("Supporting workbook inspection")
print("=" * 60)
for sheet_name, frame in supporting_sheets.items():
    print_sheet_profile(sheet_name, frame)
    print_unique_values(frame, ROUTING_COLUMNS)

interval_columns_by_sheet = {sheet_name: identify_interval_columns(frame.columns) for sheet_name, frame in supporting_sheets.items()}
print("Interval columns by sheet:")
for sheet_name, interval_cols in interval_columns_by_sheet.items():
    print(f"- {sheet_name}: {len(interval_cols)} columns")
    print(interval_cols[:8], "...", interval_cols[-8:])

print()
print("REA workbook raw preview")
print("=" * 60)
print(f"REA rows: {len(rea_reference_raw):,}")
print(f"REA columns: {len(rea_reference_raw.columns):,}")
display(rea_reference_raw.head(15))

Supporting workbook inspection
Sheet: sch_at
- rows: 23,553
- columns: 108
- interval columns detected: 96
- first columns: ['sch_date', 'to_seb', 'applicant', 'to_util', 'from_seb', 'from_util', 'appr_no', 'region', 'link', 'energy0000', 'energy0015', 'energy0030']


,sch_date,to_seb,applicant,to_util,from_seb,from_util,appr_no,region,link,energy0000,energy0015,energy0030,energy0045,energy0100,energy0115,energy0130,energy0145,energy0200,energy0215,energy0230,energy0245,energy0300,energy0315,energy0330,energy0345,energy0400,energy0415,energy0430,energy0445,energy0500,energy0515,energy0530,energy0545,energy0600,energy0615,energy0630,energy0645,energy0700,energy0715,energy0730,energy0745,energy0800,energy0815,energy0830,energy0845,energy0900,energy0915,energy0930,energy0945,energy1000,energy1015,energy1030,energy1045,energy1100,energy1115,energy1130,energy1145,energy1200,energy1215,energy1230,energy1245,energy1300,energy1315,energy1330,energy1345,energy1400,energy1415,energy1430,energy1445,energy1500,energy1515,energy1530,energy1545,energy1600,energy1615,energy1630,energy1645,energy1700,energy1715,energy1730,energy1745,energy1800,energy1815,energy1830,energy1845,energy1900,energy1915,energy1930,energy1945,energy2000,energy2015,energy2030,energy2045,energy2100,energy2115,energy2130,energy2145,energy2200,energy2215,energy2230,energy2245,energy2300,energy2315,energy2330,energy2345,total_energy,op_flag,created_on
0,2026-02-01,CH,AWEK1L,CH,NaN,AWEK1L,WR/01102023/31012046/L_NR_2021_07,2,WR-NR,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.000325,0.00015,0.00015,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.000150,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00015,0.00480,0.00560,0.006875,0.007675,0.007675,0.00785,0.00735,0.007525,0.07695,NaN,NaN
1,2026-02-01,CH,AWEK1L,CH,NaN,AWEK1L,WR/01102023/31012046/L_NR_2021_08,2,WR-NR,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.000475,0.00025,0.00025,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.000250,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00025,0.00720,0.00840,0.010325,0.011525,0.011525,0.01175,0.01105,0.011275,0.11585,NaN,NaN
2,2026-02-01,CH,KHURJA_STPP_UP,CPDL_UT,UP,KHURJA_STPP_UP,GNA/NR/2026/01/01/1886,1,NR-NR,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.022650,0.000000,0.000000,0.000000,0.000000,0.000000,0.01245,0.01200,0.007075,0.007075,0.011225,0.011225,0.011225,0.011225,0.011225,0.011225,0.007075,0.007075,0.007075,0.007075,0.007075,0.007075,0.007075,0.007075,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.02265,0.022650,0.022650,0.022650,0.02265,0.02265,0.000000,1.79335,NaN,NaN



Unique values for from_seb (36 shown up to 40):
['AMNSI', 'ANDHR', 'ARUN', 'ASM', 'BHT', 'BIHAR', 'CH', 'CHTIS', 'DL', 'DNH', 'DVC', 'GOA', 'GUJ', 'HP', 'HR', 'JHKD', 'JK', 'KAR', 'KRL', 'MANP', 'MEG', 'MH', 'MP', 'NEP', 'NEPU', 'NFL', 'ORSA', 'PB', 'POND', 'RJ', 'SIKM', 'TLGNA', 'TMNDU', 'UKD', 'UP', 'WB']

Unique values for to_seb (22 shown up to 40):
['CH', 'DL', 'GHPBR', 'GHPC1', 'GHPC2', 'GHPC3', 'GHPKL', 'GHPNJ', 'GHPP2', 'GHPRM', 'HP', 'HR', 'JK', 'NEP', 'NEPU', 'NFL', 'PB', 'PGR', 'RJ', 'RLY', 'UKD', 'UP']

Unique values for from_util (397 shown up to 40):
['AAPL_BKN2', 'ABLWM1_AEROCITY', 'ACL_PSS3_KPS1_S', 'ACL_PSS4_KPS1_HS', 'ACL_PSS4_KPS1_W', 'ADPPL_FTG', 'ADSPPL_FTG', 'AEG03_S_FTG4', 'AEG08_S_FTG4', 'AEG08_W_FTG4', 'AEML', 'AEML_SEEPZ_LTD', 'AGARWAL_406AP', 'AGE25BL_PSS9_KPS1_HW', 'AGE26AL_PSS6_KPS1_S', 'AHEJ2L_S_FTG2', 'AHEJ2L_W_FTG2', 'AHEJ3L_S_FTG2', 'AHEJ3L_W_FTG2', 'AHEJ5L_PSS5_KPS1_HW', 'AHEJ5L_PSS5_KPS1_S', 'AHEJOL_S_FTG2', 'AHEJOL_W_FTG2', 'AIA_ENGG_LTD_GJ', 'AIA_L

,sch_date,link,applicant,from_seb,from_util,to_seb,to_util,appr_no,region,energy0000,energy0015,energy0030,energy0045,energy0100,energy0115,energy0130,energy0145,energy0200,energy0215,energy0230,energy0245,energy0300,energy0315,energy0330,energy0345,energy0400,energy0415,energy0430,energy0445,energy0500,energy0515,energy0530,energy0545,energy0600,energy0615,energy0630,energy0645,energy0700,energy0715,energy0730,energy0745,energy0800,energy0815,energy0830,energy0845,energy0900,energy0915,energy0930,energy0945,energy1000,energy1015,energy1030,energy1045,energy1100,energy1115,energy1130,energy1145,energy1200,energy1215,energy1230,energy1245,energy1300,energy1315,energy1330,energy1345,energy1400,energy1415,energy1430,energy1445,energy1500,energy1515,energy1530,energy1545,energy1600,energy1615,energy1630,energy1645,energy1700,energy1715,energy1730,energy1745,energy1800,energy1815,energy1830,energy1845,energy1900,energy1915,energy1930,energy1945,energy2000,energy2015,energy2030,energy2045,energy2100,energy2115,energy2130,energy2145,energy2200,energy2215,energy2230,energy2245,energy2300,energy2315,energy2330,energy2345,total_energy,op_flag,created_on,idnum
0,2026-02-01,ER-NR,NaN,NR,IEX,ER,ER,NaN,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,21446282
1,2026-02-01,ER-NR,NaN,NR,PXI,ER,ER,NaN,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,21446283
2,2026-02-01,ER-NR,NaN,NR,HPX,ER,ER,NaN,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,21446284



Unique values for from_seb (27 shown up to 40):
['AMNSI', 'ANDHR', 'ARUN', 'ASM', 'BGDSH', 'BHT', 'BIHAR', 'CHTIS', 'DNH', 'DVC', 'GOA', 'GUJ', 'JHKD', 'KAR', 'KRL', 'MANP', 'MEG', 'MH', 'MP', 'NEP', 'NR', 'ORSA', 'POND', 'SIKM', 'TLGNA', 'TMNDU', 'WB']

Unique values for to_seb (13 shown up to 40):
['CH', 'DL', 'ER', 'HP', 'HR', 'JK', 'PB', 'PGR', 'RJ', 'RLY', 'UKD', 'UP', 'WR']

Unique values for from_util (320 shown up to 40):
['ACL_PSS3_KPS1_S', 'ACL_PSS4_KPS1_HS', 'ACL_PSS4_KPS1_W', 'AEML', 'AEML_SEEPZ_LTD', 'AGARWAL_406AP', 'AGE25BL_PSS9_KPS1_HW', 'AGE26AL_PSS6_KPS1_S', 'AHEJ5L_PSS5_KPS1_HW', 'AHEJ5L_PSS5_KPS1_S', 'AIA_ENGG_LTD_GJ', 'AIA_LTD_GUJ', 'ALPLAINDIAPVTLTD_TG', 'AMIT_FERRO_ALLOYS', 'AMNSIL', 'APCPDCL', 'APL_Raigarh_TPP', 'APL_Raipur_TPP', 'APL_Stage_3_U_7_8', 'APL_Stage_3_U_9', 'APNRL', 'APSEZ_PSS4_KPS1_W', 'ARE3L_PSS8_KPS3_HW', 'ARE41L_PSS4_KPS1_W', 'AWEK1L', 'AWEKFL', 'AWEMP1PL_PTNGR_IDR_W', 'AlfanarWind_SECI_III', 'Alpha_Silvassa_DNHDD', 'Anubha2025_Guj', 'Arinsun_RU

,sch_date,stn_code,from_seb,to_seb,energy0000,energy0015,energy0030,energy0045,energy0100,energy0115,energy0130,energy0145,energy0200,energy0215,energy0230,energy0245,energy0300,energy0315,energy0330,energy0345,energy0400,energy0415,energy0430,energy0445,energy0500,energy0515,energy0530,energy0545,energy0600,energy0615,energy0630,energy0645,energy0700,energy0715,energy0730,energy0745,energy0800,energy0815,energy0830,energy0845,energy0900,energy0915,energy0930,energy0945,energy1000,energy1015,energy1030,energy1045,energy1100,energy1115,energy1130,energy1145,energy1200,energy1215,energy1230,energy1245,energy1300,energy1315,energy1330,energy1345,energy1400,energy1415,energy1430,energy1445,energy1500,energy1515,energy1530,energy1545,energy1600,energy1615,energy1630,energy1645,energy1700,energy1715,energy1730,energy1745,energy1800,energy1815,energy1830,energy1845,energy1900,energy1915,energy1930,energy1945,energy2000,energy2015,energy2030,energy2045,energy2100,energy2115,energy2130,energy2145,energy2200,energy2215,energy2230,energy2245,energy2300,energy2315,energy2330,energy2345,total_schd,op_flag,created_on
0,2026-02-01,RIH1,CH,MP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.031425,0.00270,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,2026-02-01,RIH1,PGNR,MP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000075,0.000075,0.000075,0.000075,0.000075,0.000075,0.000075,0.000075,0.000075,0.000075,0.000075,0.000025,0.000025,0.000025,0.001825,0.001825,0.00015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2,2026-02-01,RIH1,UP,MP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.175,0.342225,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.339,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.01360,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN



Unique values for from_seb (12 shown up to 40):
['CH', 'DL', 'GUJ', 'HP', 'HR', 'JK', 'MP', 'PB', 'PGNR', 'RJ', 'UKD', 'UP']

Unique values for to_seb (3 shown up to 40):
['MP', 'UP', 'VE']

Column 'from_util' not present in this sheet.
Column 'to_util' not present in this sheet.
Column 'applicant' not present in this sheet.
Column 'link' not present in this sheet.
Column 'region' not present in this sheet.
Interval columns by sheet:
- sch_at: 96 columns
['energy0000', 'energy0015', 'energy0030', 'energy0045', 'energy0100', 'energy0115', 'energy0130', 'energy0145'] ... ['energy2200', 'energy2215', 'energy2230', 'energy2245', 'energy2300', 'energy2315', 'energy2330', 'energy2345']
- sch_cr: 96 columns
['energy0000', 'energy0015', 'energy0030', 'energy0045', 'energy0100', 'energy0115', 'energy0130', 'energy0145'] ... ['energy2200', 'energy2215', 'energy2230', 'energy2245', 'energy2300', 'energy2315', 'energy2330', 'energy2345']
- d_urs_one_to_one: 96 columns
['energy0000', 'energy0015',

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,"REA Report (Part - A, B and Bihar)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(As per new Regulation effective from 01.04.2020),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Download Excel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Tidy transformation

The schedule workbook is wide at the interval level. The next step reshapes the ENERGY columns to long format, keeps the row-level routing fields intact, and calculates daily row totals.

Important note: the notebook labels the interval field as `energy_mwh` because the source columns are named `ENERGY####`, but that unit interpretation should still be confirmed against the source methodology.

In [5]:
def collect_plant_like_names(schedule_rows: pd.DataFrame, rea_reference_raw: pd.DataFrame) -> list[str]:
    candidates: set[str] = set()

    for column in PLANT_CANDIDATE_COLUMNS:
        if column not in schedule_rows.columns:
            continue
        values = schedule_rows[column].dropna().astype("string").str.strip().replace("", pd.NA).dropna().tolist()
        for value in values:
            value_upper = standardize_text(value) or ""
            if len(value_upper) <= 2 and value_upper not in {"KB", "CH"}:
                continue
            if value_upper in {"ER", "WR", "NR", "SR", "NER", "IEX", "PXI", "PXIL", "HPX", "NA"}:
                continue
            if any(keyword in value_upper for keyword in ["BARH", "BARAUNI", "KBUNL", "NABINAGAR", "NPGC", "BRBCL", "BUXAR", "BSHPC", "GANDAK", "TRIVENI", "SONE", "BIOMASS", "SUGAR", "DISTILLERY"]):
                candidates.add(value)

    if rea_reference_raw is not None and not rea_reference_raw.empty:
        rea_text_series = rea_reference_raw.fillna("").astype(str).agg(" | ".join, axis=1)
        for text_row in rea_text_series:
            for focus_name in REVIEW_FOCUS_NAMES:
                if focus_name.upper() in text_row.upper():
                    candidates.add(focus_name)

    return sorted(candidates, key=lambda value: standardize_text(value) or "")



def infer_source_type(raw_name: str) -> str:
    name_upper = standardize_text(raw_name) or ""
    if any(token in name_upper for token in ["GANDAK", "TRIVENI", "SONE", "HYDRO"]):
        return "hydro_candidate"
    if any(token in name_upper for token in ["BIOMASS", "SUGAR", "DISTILLERY"]):
        return "renewable_candidate"
    if any(token in name_upper for token in ["BARH", "BARAUNI", "NABINAGAR", "NPGC", "BRBCL", "BUXAR", "KBUNL"]):
        return "thermal_candidate"
    return "unreviewed_candidate"



def seed_plant_mapping(schedule_rows: pd.DataFrame, rea_reference_raw: pd.DataFrame) -> pd.DataFrame:
    detected_candidates = collect_plant_like_names(schedule_rows, rea_reference_raw)
    seeded_rows: list[dict[str, object]] = []
    detected_keys = {standardize_text(name) for name in detected_candidates}

    for raw_name in detected_candidates:
        seeded_rows.append(
            {
                "raw_name": raw_name,
                "standard_name": raw_name,
                "is_bihar_located": pd.NA,
                "is_north_bihar_located": pd.NA,
                "source_type": infer_source_type(raw_name),
                "mapping_confidence": "unreviewed",
                "notes": "Detected in workbook text. Review before treating as Bihar-located generation.",
            }
        )

    for raw_name in REVIEW_FOCUS_NAMES:
        key = standardize_text(raw_name)
        if key not in detected_keys:
            seeded_rows.append(
                {
                    "raw_name": raw_name,
                    "standard_name": raw_name,
                    "is_bihar_located": pd.NA,
                    "is_north_bihar_located": pd.NA,
                    "source_type": infer_source_type(raw_name),
                    "mapping_confidence": "needs_file_confirmation",
                    "notes": "Review-focus candidate from the brief. Not directly detected yet in the loaded workbook text.",
                }
            )

    return pd.DataFrame(seeded_rows)



def apply_entity_mapping(schedule_rows: pd.DataFrame, entity_mapping: pd.DataFrame) -> pd.DataFrame:
    mapped = schedule_rows.copy()
    mapping = entity_mapping.copy()
    mapping["entity_key"] = mapping["raw_entity"].map(standardize_text)

    source_mapping = mapping.add_suffix("_source")
    destination_mapping = mapping.add_suffix("_destination")

    mapped["source_entity_key"] = mapped["source_entity_raw"].map(standardize_text)
    mapped["destination_entity_key"] = mapped["destination_entity_raw"].map(standardize_text)

    mapped = mapped.merge(source_mapping, how="left", left_on="source_entity_key", right_on="entity_key_source")
    mapped = mapped.merge(destination_mapping, how="left", left_on="destination_entity_key", right_on="entity_key_destination")

    for column in [
        "belongs_to_bihar_source",
        "belongs_to_nbpdcl_source",
        "belongs_to_sbpdcl_source",
        "belongs_to_interstate_source",
        "belongs_to_bihar_destination",
        "belongs_to_nbpdcl_destination",
        "belongs_to_sbpdcl_destination",
        "belongs_to_interstate_destination",
    ]:
        if column in mapped.columns:
            mapped[column] = mapped[column].astype("boolean")

    mapped["link_text"] = coalesce_first_non_blank(mapped, ["link"])
    mapped["region_text"] = coalesce_first_non_blank(mapped, ["region"])
    mapped["from_seb_key"] = mapped.get("from_seb", pd.Series(pd.NA, index=mapped.index)).map(standardize_text)
    mapped["to_seb_key"] = mapped.get("to_seb", pd.Series(pd.NA, index=mapped.index)).map(standardize_text)

    mapped["route_cross_border_hint"] = (
        mapped["link_text"].fillna("").astype(str).str.contains("-", regex=False)
        | mapped["region_text"].fillna("").astype(str).str.contains("-", regex=False)
        | (mapped["from_seb_key"].notna() & mapped["to_seb_key"].notna() & mapped["from_seb_key"].ne(mapped["to_seb_key"]))
    )

    source_is_bihar = mapped["belongs_to_bihar_source"].fillna(False)
    destination_is_bihar = mapped["belongs_to_bihar_destination"].fillna(False)
    mapped["classification_strength"] = np.where(mapped["route_cross_border_hint"], "strong", "weak")
    mapped["route_label"] = np.select(
        [source_is_bihar & destination_is_bihar, (~source_is_bihar) & destination_is_bihar, source_is_bihar & (~destination_is_bihar)],
        ["bihar_internal_candidate", "bihar_import_candidate", "bihar_export_candidate"],
        default="unclassified",
    )
    return mapped



def build_sign_diagnostics(mapped_rows: pd.DataFrame) -> pd.DataFrame:
    diagnostics = (
        mapped_rows.assign(
            positive_row=lambda df: df["daily_total_raw"] > 0,
            negative_row=lambda df: df["daily_total_raw"] < 0,
            zero_or_missing=lambda df: df["daily_total_raw"].isna() | df["daily_total_raw"].eq(0),
        )
        .groupby(["route_label", "classification_strength"], dropna=False)
        .agg(
            rows=("row_uid", "nunique"),
            positive_rows=("positive_row", "sum"),
            negative_rows=("negative_row", "sum"),
            zero_or_missing_rows=("zero_or_missing", "sum"),
            raw_total_mwh=("daily_total_raw", "sum"),
            abs_total_mwh=("daily_total_abs", "sum"),
        )
        .reset_index()
        .sort_values(["route_label", "classification_strength"])
    )
    return diagnostics



def choose_sign_convention(diagnostics: pd.DataFrame) -> tuple[str, str]:
    imports = diagnostics.loc[diagnostics["route_label"] == "bihar_import_candidate"]
    exports = diagnostics.loc[diagnostics["route_label"] == "bihar_export_candidate"]

    import_positive = imports["positive_rows"].sum()
    import_negative = imports["negative_rows"].sum()
    export_positive = exports["positive_rows"].sum()
    export_negative = exports["negative_rows"].sum()

    if import_positive > import_negative and export_negative > export_positive:
        return "positive_import_negative_export", "Import rows are mostly positive and export rows are mostly negative, so imports keep the raw sign and exports are flipped to positive reporting magnitudes."
    if import_negative > import_positive and export_positive > export_negative:
        return "negative_import_positive_export", "Import rows are mostly negative and export rows are mostly positive, so imports are sign-flipped and exports keep the raw sign."
    return "direction_from_routing_use_absolute_magnitude", "Routing fields carry the cleaner directional evidence, but the numeric sign is not consistently directional. Import and export magnitudes therefore use absolute values after route selection."



def add_effective_energy(frame: pd.DataFrame, sign_mode: str, value_column: str = "daily_total_raw") -> pd.DataFrame:
    enriched = frame.copy()
    if sign_mode == "positive_import_negative_export":
        enriched["effective_energy_mwh"] = np.select(
            [enriched["route_label"].eq("bihar_import_candidate"), enriched["route_label"].eq("bihar_export_candidate")],
            [enriched[value_column], -enriched[value_column]],
            default=enriched[value_column].abs(),
        )
    elif sign_mode == "negative_import_positive_export":
        enriched["effective_energy_mwh"] = np.select(
            [enriched["route_label"].eq("bihar_import_candidate"), enriched["route_label"].eq("bihar_export_candidate")],
            [-enriched[value_column], enriched[value_column]],
            default=enriched[value_column].abs(),
        )
    else:
        enriched["effective_energy_mwh"] = enriched[value_column].abs()
    return enriched



def apply_plant_mapping(schedule_rows: pd.DataFrame, plant_mapping: pd.DataFrame) -> pd.DataFrame:
    mapped = schedule_rows.copy()
    mapping = plant_mapping.copy()
    mapping["plant_key"] = mapping["raw_name"].map(standardize_text)
    plant_keys = set(mapping["plant_key"].dropna())

    mapped["plant_raw_name"] = pd.NA
    for column in PLANT_CANDIDATE_COLUMNS:
        if column not in mapped.columns:
            continue
        candidate_key = mapped[column].map(standardize_text)
        mask = candidate_key.isin(plant_keys) & mapped["plant_raw_name"].isna()
        mapped.loc[mask, "plant_raw_name"] = mapped.loc[mask, column]

    mapped["plant_key"] = mapped["plant_raw_name"].map(standardize_text)
    return mapped.merge(mapping, how="left", on="plant_key", suffixes=("", "_plant_map"))

In [6]:
prepared_row_frames: list[pd.DataFrame] = []
prepared_long_frames: list[pd.DataFrame] = []

for sheet_name, frame in supporting_sheets.items():
    row_frame, long_frame = prepare_schedule_sheet(frame, sheet_name)
    prepared_row_frames.append(row_frame)
    prepared_long_frames.append(long_frame)

schedule_rows = pd.concat(prepared_row_frames, ignore_index=True)
schedule_long = pd.concat(prepared_long_frames, ignore_index=True)

print(f"Row-level schedule table shape: {schedule_rows.shape}")
print(f"Long interval table shape: {schedule_long.shape}")
print()
print("Row-level preview:")
display(schedule_rows.head(5))
print()
print("Long-format preview:")
display(schedule_long[["row_uid", "source_sheet", "sch_date", "interval_label", "interval_start", "energy_mwh", "source_entity_raw", "destination_entity_raw"]].head(8))

schedule_long_quality = pd.DataFrame(
    {
        "rows": [len(schedule_rows)],
        "rows_with_missing_sch_date": [int(schedule_rows["sch_date"].isna().sum())],
        "interval_records": [len(schedule_long)],
        "interval_records_missing_energy": [int(schedule_long["energy_mwh"].isna().sum())],
    }
)
print("Tidy transformation diagnostics:")
display(schedule_long_quality)

AttributeError: 'DataFrame' object has no attribute 'dtype'

## 4. Mapping framework

The following tables are intentionally editable working tables, not final truth tables.

- `mapping_entity_candidates` is built from routing fields such as `from_seb`, `to_seb`, `from_util`, `to_util`, and `Applicant`.
- `mapping_plant_candidates` is seeded from plant-like names found in the loaded files, then padded with review-focus names from the brief when those names were not directly detected.

Any Bihar-located or North-Bihar-located flag should be reviewed manually before it is treated as authoritative.

In [ ]:
mapping_entity_candidates = seed_entity_mapping(schedule_rows)
mapping_plant_candidates = seed_plant_mapping(schedule_rows, rea_reference_raw)

print("Entity mapping candidates:")
display(mapping_entity_candidates.head(50))
print()
print("Plant mapping candidates:")
display(mapping_plant_candidates.head(50))

mapping_entity_candidates.to_csv(OUTPUT_DIR / "mapping_entity_candidates.csv", index=False)
mapping_plant_candidates.to_csv(OUTPUT_DIR / "mapping_plant_candidates.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'mapping_entity_candidates.csv'}")
print(f"Saved: {OUTPUT_DIR / 'mapping_plant_candidates.csv'}")

In [ ]:
def build_daily_metric(frame: pd.DataFrame, *, metric_label: str, breakdown_column: str | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    daily = (
        frame.dropna(subset=["sch_date"])
        .groupby("sch_date", as_index=False)
        .agg(
            scheduled_energy_mwh=("effective_energy_mwh", "sum"),
            source_rows=("row_uid", "nunique"),
            ambiguous_rows=("classification_strength", lambda s: int((s == "weak").sum())),
        )
        .sort_values("sch_date")
    )
    daily["metric"] = metric_label

    breakdown = pd.DataFrame()
    if breakdown_column and breakdown_column in frame.columns:
        breakdown = (
            frame.assign(**{breakdown_column: frame[breakdown_column].fillna("<missing>")})
            .groupby(breakdown_column, as_index=False)
            .agg(scheduled_energy_mwh=("effective_energy_mwh", "sum"), rows=("row_uid", "nunique"))
            .sort_values("scheduled_energy_mwh", ascending=False)
        )
    return daily, breakdown



def plot_daily_series(daily_frame: pd.DataFrame, title: str, value_column: str = "scheduled_energy_mwh") -> None:
    if daily_frame.empty:
        print(f"No rows available for chart: {title}")
        return
    plt.figure()
    plt.plot(daily_frame["sch_date"], daily_frame[value_column], marker="o", linewidth=1.5)
    plt.title(title)
    plt.xlabel("Schedule date")
    plt.ylabel("Scheduled energy (MWh-equivalent as represented in source ENERGY columns)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



def plot_stacked_daily(frame: pd.DataFrame, category_column: str, title: str, max_categories: int = 10) -> None:
    if frame.empty or category_column not in frame.columns:
        print(f"No rows available for stacked chart: {title}")
        return
    top_categories = frame.groupby(category_column)["effective_energy_mwh"].sum().sort_values(ascending=False).head(max_categories).index
    pivoted = (
        frame[frame[category_column].isin(top_categories)]
        .pivot_table(index="sch_date", columns=category_column, values="effective_energy_mwh", aggfunc="sum", fill_value=0)
        .sort_index()
    )
    if pivoted.empty:
        print(f"No pivot rows available for stacked chart: {title}")
        return
    ax = pivoted.plot(kind="bar", stacked=True, figsize=(12, 5))
    ax.set_title(title)
    ax.set_xlabel("Schedule date")
    ax.set_ylabel("Scheduled energy (MWh-equivalent)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



def save_csv(frame: pd.DataFrame, path: Path) -> None:
    frame.to_csv(path, index=False)
    print(f"Saved: {path}")



def save_parquet(frame: pd.DataFrame, path: Path) -> None:
    frame.to_parquet(path, index=False)
    print(f"Saved: {path}")

## 5. Sign convention and routing diagnostics

The notebook does not assume that positive or negative interval values automatically mean import or export. Instead it:

1. classifies rows using routing information first
2. compares sign behaviour across Bihar-origin and Bihar-destination rows
3. chooses a sign convention only if the evidence is consistent
4. otherwise reports import / export magnitudes using absolute values after route selection

This keeps the direction logic visible and avoids silently forcing a sign rule.

In [ ]:
mapped_rows = apply_entity_mapping(schedule_rows, mapping_entity_candidates)
sign_diagnostics = build_sign_diagnostics(mapped_rows)
chosen_sign_mode, chosen_sign_explanation = choose_sign_convention(sign_diagnostics)
metric_rows = add_effective_energy(mapped_rows, chosen_sign_mode)

print("Routing diagnostics:")
display(sign_diagnostics)
print()
display(Markdown(f"**Chosen sign convention**: {chosen_sign_explanation}"))

route_preview_columns = ["source_sheet", "sch_date", "source_entity_raw", "destination_entity_raw", "from_seb", "to_seb", "link_text", "region_text", "route_cross_border_hint", "classification_strength", "route_label", "daily_total_raw", "effective_energy_mwh"]
display(metric_rows[route_preview_columns].head(20))

## 6. Metric calculations

The core daily outputs are built next:

1. daily scheduled energy generated within Bihar, where the plant mapping explicitly supports Bihar-located generation
2. daily scheduled interstate-to-Bihar imports
3. daily scheduled Bihar-to-interstate exports

Where a clean derivation is not possible, the notebook keeps the result provisional and shows the row set behind it.

In [ ]:
plant_mapped_rows = apply_plant_mapping(metric_rows, mapping_plant_candidates)

confirmed_bihar_generation_rows = plant_mapped_rows[plant_mapped_rows["is_bihar_located"].fillna(False)].copy()
if confirmed_bihar_generation_rows.empty:
    generation_status_note = "No plant mapping rows are currently marked `is_bihar_located == True`. Generation output below is provisional and limited to candidate rows that matched the plant mapping table."
    generation_rows = plant_mapped_rows[plant_mapped_rows["plant_raw_name"].notna()].copy()
    generation_rows["metric_status"] = "provisional"
else:
    generation_status_note = "Generation output uses plant rows explicitly marked `is_bihar_located == True` in the editable mapping table."
    generation_rows = confirmed_bihar_generation_rows.copy()
    generation_rows["metric_status"] = "reviewed_candidate"

generation_rows["effective_energy_mwh"] = generation_rows["effective_energy_mwh"].abs()
generation_rows["plant_display_name"] = generation_rows["standard_name"].fillna(generation_rows["plant_raw_name"]).fillna("<unmapped plant>")
daily_bihar_generation, generation_by_plant = build_daily_metric(generation_rows, metric_label="daily_scheduled_energy_generated_within_bihar", breakdown_column="plant_display_name")

print(generation_status_note)
display(daily_bihar_generation.head(20))
display(generation_by_plant.head(20))
plot_daily_series(daily_bihar_generation, "Daily Bihar scheduled generation candidate total")
plot_stacked_daily(generation_rows, "plant_display_name", "Daily Bihar scheduled generation candidate split by plant")

imports_rows = metric_rows[metric_rows["route_label"].eq("bihar_import_candidate")].copy()
imports_rows["source_breakdown"] = imports_rows["source_entity_raw"].fillna(imports_rows["from_seb"]).fillna(imports_rows["region_text"]).fillna("<missing>")
imports_rows["source_region_breakdown"] = imports_rows["region_text"].fillna(imports_rows["link_text"]).fillna("<missing>")
daily_interstate_to_bihar, imports_by_source = build_daily_metric(imports_rows, metric_label="daily_scheduled_interstate_to_bihar", breakdown_column="source_breakdown")
imports_by_region = imports_rows.groupby("source_region_breakdown", as_index=False).agg(scheduled_energy_mwh=("effective_energy_mwh", "sum"), rows=("row_uid", "nunique")).sort_values("scheduled_energy_mwh", ascending=False)

display(daily_interstate_to_bihar.head(20))
display(imports_by_source.head(20))
display(imports_by_region.head(20))
plot_daily_series(daily_interstate_to_bihar, "Daily scheduled imports into Bihar")

exports_rows = metric_rows[metric_rows["route_label"].eq("bihar_export_candidate")].copy()
exports_rows["destination_breakdown"] = exports_rows["destination_entity_raw"].fillna(exports_rows["to_seb"]).fillna(exports_rows["region_text"]).fillna("<missing>")
exports_rows["destination_region_breakdown"] = exports_rows["region_text"].fillna(exports_rows["link_text"]).fillna("<missing>")
daily_bihar_to_interstate, exports_by_destination = build_daily_metric(exports_rows, metric_label="daily_scheduled_bihar_to_interstate", breakdown_column="destination_breakdown")
exports_by_region = exports_rows.groupby("destination_region_breakdown", as_index=False).agg(scheduled_energy_mwh=("effective_energy_mwh", "sum"), rows=("row_uid", "nunique")).sort_values("scheduled_energy_mwh", ascending=False)

display(daily_bihar_to_interstate.head(20))
display(exports_by_destination.head(20))
display(exports_by_region.head(20))
plot_daily_series(daily_bihar_to_interstate, "Daily scheduled exports from Bihar")

## 7. Optional NBPDCL / SBPDCL split

This section is intentionally conservative. It only produces NBPDCL or SBPDCL daily views when the routing fields directly support those tags through the entity mapping table.

It does not attempt to infer a North Bihar geographic generation metric from allocation or tariff context alone.

In [ ]:
nbpdcl_rows = metric_rows[metric_rows["belongs_to_nbpdcl_source"].fillna(False) | metric_rows["belongs_to_nbpdcl_destination"].fillna(False)].copy()
sbpdcl_rows = metric_rows[metric_rows["belongs_to_sbpdcl_source"].fillna(False) | metric_rows["belongs_to_sbpdcl_destination"].fillna(False)].copy()

if nbpdcl_rows.empty:
    daily_nbpdcl_optional = pd.DataFrame()
    print("NBPDCL split not produced because no explicit NBPDCL routing tags were found in the current entity mapping.")
else:
    nbpdcl_rows["nbpdcl_direction"] = np.where(nbpdcl_rows["belongs_to_nbpdcl_destination"].fillna(False), "into_nbpdcl", "from_nbpdcl")
    daily_nbpdcl_optional = nbpdcl_rows.groupby(["sch_date", "nbpdcl_direction"], as_index=False).agg(scheduled_energy_mwh=("effective_energy_mwh", "sum"), rows=("row_uid", "nunique")).sort_values(["sch_date", "nbpdcl_direction"])
    display(daily_nbpdcl_optional.head(20))
    plot_daily_series(daily_nbpdcl_optional.groupby("sch_date", as_index=False)["scheduled_energy_mwh"].sum(), "Daily NBPDCL-tagged scheduled flows")

if sbpdcl_rows.empty:
    daily_sbpdcl_optional = pd.DataFrame()
    print("SBPDCL split not produced because no explicit SBPDCL routing tags were found in the current entity mapping.")
else:
    sbpdcl_rows["sbpdcl_direction"] = np.where(sbpdcl_rows["belongs_to_sbpdcl_destination"].fillna(False), "into_sbpdcl", "from_sbpdcl")
    daily_sbpdcl_optional = sbpdcl_rows.groupby(["sch_date", "sbpdcl_direction"], as_index=False).agg(scheduled_energy_mwh=("effective_energy_mwh", "sum"), rows=("row_uid", "nunique")).sort_values(["sch_date", "sbpdcl_direction"])
    display(daily_sbpdcl_optional.head(20))
    plot_daily_series(daily_sbpdcl_optional.groupby("sch_date", as_index=False)["scheduled_energy_mwh"].sum(), "Daily SBPDCL-tagged scheduled flows")

## 8. Monthly reconciliation

The REA workbook is treated as a monthly reference rather than the main daily numerical source. Because REA workbooks are often semi-formatted, this notebook uses a best-effort text-and-numeric scan to identify Bihar- or station-related rows for comparison.

Any reconciliation mismatch here should be interpreted as a review cue, not as proof that the daily schedule extraction is wrong.

## 9. Outputs

The final cell saves the requested CSV outputs into `data/processed/bihar_energy_flow/` and also persists reusable parquet tables for the row-level and interval-long schedule data.

## 10. Visual output

Separate matplotlib charts are created during the metric sections above. The notebook intentionally avoids seaborn and subplot-heavy layouts so each view stays easy to inspect and edit.

In [ ]:
rea_row_text = rea_reference_raw.fillna("").astype(str).agg(" | ".join, axis=1)
rea_keywords = ["BIHAR", "NBPDCL", "SBPDCL", "BARH", "BARAUNI", "KBUNL", "NABINAGAR", "NPGC", "BRBCL", "BUXAR", "BSHPC", "GANDAK", "TRIVENI", "SONE"]
rea_keyword_mask = rea_row_text.str.contains("|".join(rea_keywords), case=False, na=False)
rea_candidate_rows = rea_reference_raw.loc[rea_keyword_mask].copy()
rea_candidate_numeric = rea_candidate_rows.apply(pd.to_numeric, errors="coerce")
rea_candidate_numeric_sum = rea_candidate_numeric.sum().dropna().sum() if not rea_candidate_numeric.empty else np.nan

monthly_comparison = pd.DataFrame([
    {"metric": "daily_scheduled_energy_generated_within_bihar", "notebook_month_total_mwh": daily_bihar_generation["scheduled_energy_mwh"].sum() if not daily_bihar_generation.empty else np.nan, "rea_reference_context": "best_effort_bihar_or_station_keyword_scan", "rea_reference_numeric_cell_sum": rea_candidate_numeric_sum, "comment": generation_status_note},
    {"metric": "daily_scheduled_interstate_to_bihar", "notebook_month_total_mwh": daily_interstate_to_bihar["scheduled_energy_mwh"].sum() if not daily_interstate_to_bihar.empty else np.nan, "rea_reference_context": "best_effort_bihar_or_station_keyword_scan", "rea_reference_numeric_cell_sum": rea_candidate_numeric_sum, "comment": "Compare with REA monthly import-related rows only after manual REA structure review."},
    {"metric": "daily_scheduled_bihar_to_interstate", "notebook_month_total_mwh": daily_bihar_to_interstate["scheduled_energy_mwh"].sum() if not daily_bihar_to_interstate.empty else np.nan, "rea_reference_context": "best_effort_bihar_or_station_keyword_scan", "rea_reference_numeric_cell_sum": rea_candidate_numeric_sum, "comment": "Compare with REA monthly export-related rows only after manual REA structure review."},
])

print("REA candidate rows used for best-effort monthly reconciliation:")
display(rea_candidate_rows.head(20))
print("Monthly comparison summary:")
display(monthly_comparison)

save_csv(daily_bihar_generation, OUTPUT_DIR / "daily_bihar_generation.csv")
save_csv(daily_interstate_to_bihar, OUTPUT_DIR / "daily_interstate_to_bihar.csv")
save_csv(daily_bihar_to_interstate, OUTPUT_DIR / "daily_bihar_to_interstate.csv")
if not daily_nbpdcl_optional.empty:
    save_csv(daily_nbpdcl_optional, OUTPUT_DIR / "daily_nbpdcl_optional.csv")
if not daily_sbpdcl_optional.empty:
    save_csv(daily_sbpdcl_optional, OUTPUT_DIR / "daily_sbpdcl_optional.csv")

save_parquet(schedule_rows, OUTPUT_DIR / "bihar_schedule_rows.parquet")
save_parquet(schedule_long, OUTPUT_DIR / "bihar_schedule_long.parquet")

print()
print("Output directory contents:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(f"- {path.name}")